# Challenge 2: Will the visit end in a purchase?

Each row is one visit to an online store. We predict `Revenue`: whether the visit ended in a purchase (True/False). This is binary classification.

This notebook covers the first part of the validation strategy: loading the data and splitting it into training and hold-out sets with the same share of buyers in each.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold

RANDOM_STATE = 42

## Load the data

In [2]:
train = pd.read_csv("challenge2_train.csv")

print(train.shape)
train.head()

(9247, 19)


,ID,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,1,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,3,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
2,4,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
3,7,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.4,Feb,2,4,3,3,Returning_Visitor,False,False
4,9,0,0.0,0,0.0,2,37.000000,0.00,0.10,0.0,0.8,Feb,2,2,2,3,Returning_Visitor,False,False


In [3]:
# column types and missing values
summary = pd.DataFrame({
    "type": train.dtypes.astype(str),
    "missing": train.isna().sum(),
    "distinct values": train.nunique(),
})
summary

,type,missing,distinct values
ID,int64,0,9247
Administrative,int64,0,26
Administrative_Duration,float64,0,2677
Informational,int64,0,17
Informational_Duration,float64,0,1022
ProductRelated,int64,0,278
ProductRelated_Duration,float64,0,7336
BounceRates,float64,0,1506
ExitRates,float64,0,3748
PageValues,float64,0,2042


## Target and features

`Revenue` and `Weekend` can be read in as True/False or as text, so we turn both into 1/0. `ID` is only a row label, so it is dropped.

In [4]:
def to_01(col):
    """Turn True/False (bool or text) into 1/0."""
    if col.dtype == bool:
        return col.astype(int)
    return col.astype(str).str.strip().str.lower().map({"true": 1, "false": 0})

y = to_01(train["Revenue"])
X = train.drop(columns=["ID", "Revenue"]).copy()
X["Weekend"] = to_01(X["Weekend"])

assert y.notna().all(), "Revenue has values that are not True/False"
assert X["Weekend"].notna().all(), "Weekend has values that are not True/False"

print(f"Buyers: {y.sum():,} of {len(y):,} visits ({y.mean():.1%})")

Buyers: 1,431 of 9,247 visits (15.5%)


Only a small share of visits end in a purchase. That means:

- Accuracy is misleading: always guessing "no purchase" already scores high.
- Every split has to keep the same share of buyers, or some folds get too few to learn from or to score on. That is what `stratify=y` and `StratifiedKFold` do.

## Split

- **Hold-out (20%)**: set aside and not touched until the final model is chosen. It is scored once.
- **5-fold cross validation** on the other 80%: used to compare models, check for overfitting, and tune settings. Every model uses these same folds so the scores are comparable.

In [5]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## Check the buyer share in every piece

In [6]:
split_check = pd.DataFrame({
    "rows":        [len(y), len(y_train), len(y_val)],
    "buyers":      [y.sum(), y_train.sum(), y_val.sum()],
    "buyer share": [y.mean(), y_train.mean(), y_val.mean()],
}, index=["all data", "train (80%)", "hold-out (20%)"])
split_check.style.format({"buyer share": "{:.1%}"})

,rows,buyers,buyer share
all data,9247,1431,15.5%
train (80%),7397,1145,15.5%
hold-out (20%),1850,286,15.5%


In [7]:
fold_check = pd.DataFrame([
    {
        "fold": i,
        "train rows": len(tr),
        "train buyer share": y_train.iloc[tr].mean(),
        "scoring rows": len(va),
        "scoring buyer share": y_train.iloc[va].mean(),
    }
    for i, (tr, va) in enumerate(cv.split(X_train, y_train), 1)
]).set_index("fold")
fold_check.style.format({"train buyer share": "{:.1%}", "scoring buyer share": "{:.1%}"})

,train rows,train buyer share,scoring rows,scoring buyer share
fold,,,,
1,5917,15.5%,1480,15.5%
2,5917,15.5%,1480,15.5%
3,5918,15.5%,1479,15.5%
4,5918,15.5%,1479,15.5%
5,5918,15.5%,1479,15.5%


In [8]:
# every piece should be within half a percentage point of the full data
shares = list(split_check["buyer share"]) + list(fold_check["train buyer share"]) + list(fold_check["scoring buyer share"])
assert max(abs(s - y.mean()) for s in shares) < 0.005
assert len(set(X_train.index) & set(X_val.index)) == 0, "hold-out rows leaked into training"
print("Splits are stratified and the hold-out does not overlap training.")

Splits are stratified and the hold-out does not overlap training.


## How to use this in the next steps

- Compare models with `cross_validate(model, X_train, y_train, cv=cv, ...)` using these exact folds.
- Tune settings with `GridSearchCV(..., cv=cv)` fitted on `X_train, y_train` only.
- Score `X_val, y_val` once, after the final model is chosen.
- Refit the final model on all of `X, y` before predicting the test set.